# Equation → Verilog RTL Compiler

**SC-NeuroCore v3.14** — Define neuron dynamics as ODE strings, simulate in Python,
then compile the same equations to synthesisable Q8.8 fixed-point Verilog.

This notebook demonstrates the complete equation-to-hardware pipeline:

1. **Define** neuron ODEs as Brian2-style strings
2. **Simulate** in Python with `EquationNeuron.step()`
3. **Compile** to Q8.8 Verilog with `compile_to_verilog()`
4. **Compare** multi-variable models (FitzHugh-Nagumo, Izhikevich)
5. **Inspect** the generated RTL

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.neurons.equation_builder import EquationNeuron, from_equations
from sc_neurocore.compiler.equation_compiler import compile_to_verilog, Q88

print("SC-NeuroCore equation-to-Verilog demo")

## 1. Define a LIF Neuron from Strings

The `from_equations()` factory parses Brian2-style ODE strings:

$$\tau_m \frac{dV}{dt} = -(V - E_L) + R \cdot I$$

Written as `dv/dt = -(v - E_L)/tau_m + I/C`. The parser extracts
the variable name and right-hand side, compiles to Python bytecode
for simulation, and stores the AST for Verilog emission.

In [ ]:
lif = from_equations(
    "dv/dt = -(v - E_L)/tau_m + I/C",
    threshold="v > V_th",
    reset="v = V_reset",
    params=dict(E_L=-65.0, tau_m=10.0, C=1.0),
    init=dict(v=-65.0),
    dt=0.1,
)
lif.constants = {"V_th": -50.0, "V_reset": -65.0}

print(f"State variables: {list(lif.equations.keys())}")
print(f"Equations:       {lif.equations}")
print(f"Threshold:       {lif.threshold_expr}")
print(f"Reset:           {lif.reset_rules}")
print(f"Parameters:      {lif.parameters}")

## 2. Simulate in Python

The `EquationNeuron.step()` method uses Euler integration on the
compiled expressions. This is the golden reference for hardware
co-simulation.

In [ ]:
T = 2000  # steps at dt=0.1 ms → 200 ms
voltages = np.zeros(T)
spikes = []

# Step current: 0 nA for 50 ms, then 15 nA
for t in range(T):
    I = 0.0 if t < 500 else 15.0
    spike = lif.step(I=I)
    voltages[t] = lif.state["v"]
    if spike:
        spikes.append(t * 0.1)

time_ms = np.arange(T) * 0.1

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_ms, voltages, linewidth=0.8)
ax.axhline(-50, color="r", linestyle="--", alpha=0.4, label="threshold")
ax.axvline(50, color="gray", linestyle=":", alpha=0.5, label="stimulus onset")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Membrane voltage (mV)")
ax.set_title(f"LIF from string equations — {len(spikes)} spikes")
ax.legend()
plt.tight_layout()
plt.show()

if spikes:
    isis = np.diff(spikes)
    print(f"Spike count: {len(spikes)}")
    print(f"Mean ISI: {np.mean(isis):.2f} ms")
    print(f"Firing rate: {1000 / np.mean(isis):.1f} Hz")

## 3. Compile to Verilog

`compile_to_verilog()` walks the equation AST and emits Q8.8
fixed-point Verilog. Each ODE term becomes a multiply-shift
pipeline stage. Parameters are Verilog `localparam` constants.

The generated module has:
- `clk`, `rst_n` — clock and active-low reset
- `I_t` — signed 16-bit input current
- `spike` — 1-bit output
- One register per state variable (`v_reg`)

In [ ]:
verilog_lif = compile_to_verilog(lif, module_name="sc_lif_from_eq")

print(f"Generated {len(verilog_lif)} characters of Verilog")
print(f"Lines: {verilog_lif.count(chr(10)) + 1}")
print()
print(verilog_lif)

## 4. FitzHugh-Nagumo: Multi-Variable ODE

The compiler handles multi-variable systems. FitzHugh-Nagumo has
two coupled ODEs — a fast voltage variable $v$ and a slow recovery
variable $w$:

$$\frac{dv}{dt} = v - \frac{v^3}{3} - w + I$$
$$\frac{dw}{dt} = \epsilon(v + a - bw)$$

The `v**3` term compiles to chained Q8.8 multiplications with
arithmetic right-shift after each stage.

In [ ]:
fhn = EquationNeuron(
    equations={
        "v": "v - v**3 / 3 - w + I",
        "w": "epsilon * (v + a - b * w)",
    },
    parameters={"epsilon": 0.08, "a": 0.7, "b": 0.8},
    state={"v": -1.0, "w": -0.5},
    threshold="v > 1.0",
    reset={"v": "-1.0"},
    dt=0.05,
)

T_fhn = 8000
v_trace = np.zeros(T_fhn)
w_trace = np.zeros(T_fhn)

for t in range(T_fhn):
    I = 0.5 if t > 1000 else 0.0
    fhn.step(I=I)
    v_trace[t] = fhn.state["v"]
    w_trace[t] = fhn.state["w"]

time_fhn = np.arange(T_fhn) * 0.05

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(time_fhn, v_trace, linewidth=0.6, label="v (fast)")
axes[0].plot(time_fhn, w_trace, linewidth=0.6, label="w (slow)")
axes[0].axvline(50, color="gray", linestyle=":", alpha=0.5)
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("State")
axes[0].set_title("FitzHugh-Nagumo Time Series")
axes[0].legend()

axes[1].plot(v_trace[1000:], w_trace[1000:], linewidth=0.4, alpha=0.7)
axes[1].set_xlabel("v")
axes[1].set_ylabel("w")
axes[1].set_title("Phase Portrait (limit cycle)")

plt.tight_layout()
plt.show()

In [ ]:
verilog_fhn = compile_to_verilog(fhn, module_name="sc_fitzhugh_nagumo")

print(f"FHN Verilog: {len(verilog_fhn)} chars, {verilog_fhn.count(chr(10)) + 1} lines")
print()
# Show first 40 lines
for i, line in enumerate(verilog_fhn.split(chr(10))[:40]):
    print(f"{i+1:3d} | {line}")

## 5. Izhikevich Model

Two-variable model with quadratic voltage dynamics (Izhikevich 2003):

$$\frac{dv}{dt} = 0.04 v^2 + 5v + 140 - u + I$$
$$\frac{du}{dt} = a(bv - u)$$

With reset: $v \leftarrow c$, $u \leftarrow u + d$ on spike.

In [ ]:
izh = EquationNeuron(
    equations={
        "v": "0.04 * v**2 + 5 * v + 140 - u + I",
        "u": "a * (b * v - u)",
    },
    parameters={"a": 0.02, "b": 0.2},
    state={"v": -65.0, "u": -14.0},
    threshold="v > 30",
    reset={"v": "-65", "u": "u + 8"},
    constants={},
    dt=0.5,
)

# Regular spiking pattern
T_izh = 2000
v_izh = np.zeros(T_izh)
u_izh = np.zeros(T_izh)
spikes_izh = []

for t in range(T_izh):
    I = 10.0 if t > 200 else 0.0
    spike = izh.step(I=I)
    v_izh[t] = izh.state["v"]
    u_izh[t] = izh.state["u"]
    if spike:
        spikes_izh.append(t * 0.5)

time_izh = np.arange(T_izh) * 0.5

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(time_izh, v_izh, linewidth=0.6)
axes[0].set_ylabel("v (mV)")
axes[0].set_title(f"Izhikevich Regular Spiking — {len(spikes_izh)} spikes")

axes[1].plot(time_izh, u_izh, linewidth=0.6, color="orange")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("u (recovery)")

plt.tight_layout()
plt.show()

In [ ]:
verilog_izh = compile_to_verilog(izh, module_name="sc_izhikevich")
print(f"Izhikevich Verilog: {len(verilog_izh)} chars, {verilog_izh.count(chr(10)) + 1} lines")
print()
for i, line in enumerate(verilog_izh.split(chr(10))[:40]):
    print(f"{i+1:3d} | {line}")

## 6. Q8.8 Fixed-Point Format

All arithmetic uses Q8.8 signed fixed-point: 8 integer bits,
8 fractional bits, range $[-128, +127.996]$, resolution $2^{-8} \approx 0.0039$.

Multiplication: `(a * b) >>> 8` — wide product followed by arithmetic
right-shift to re-normalise. Division by constant converts to
multiply-by-reciprocal.

In [ ]:
q = Q88()

# Demonstrate encoding
test_values = [-65.0, -50.0, 0.0, 0.04, 0.2, 1.0, 5.0, 30.0, 140.0]
print(f"{'Float':>10s}  {'Q8.8 int':>10s}  {'Q8.8 hex':>10s}  {'Roundtrip':>10s}  {'Error':>10s}")
print("-" * 56)
for v in test_values:
    encoded = q.encode(v)
    # Decode: interpret as signed, then divide by 256
    if encoded >= (1 << 15):
        decoded = (encoded - (1 << 16)) / 256.0
    else:
        decoded = encoded / 256.0
    err = abs(v - decoded)
    print(f"{v:10.3f}  {encoded:10d}  {encoded:10s}  {decoded:10.4f}  {err:10.4f}")

## 7. Comparison: Three Models Side by Side

All three models above — LIF, FitzHugh-Nagumo, Izhikevich — were
defined as strings, simulated in Python, and compiled to Verilog
from the same source representation.

In [ ]:
models = {
    "LIF": verilog_lif,
    "FitzHugh-Nagumo": verilog_fhn,
    "Izhikevich": verilog_izh,
}

print(f"{'Model':<20s}  {'Lines':>6s}  {'Chars':>7s}  {'Registers':>10s}  {'Multipliers':>12s}")
print("-" * 60)
for name, code in models.items():
    n_lines = code.count("\n") + 1
    n_chars = len(code)
    n_regs = code.count("_reg")
    n_muls = code.count("_mul")
    print(f"{name:<20s}  {n_lines:6d}  {n_chars:7d}  {n_regs:10d}  {n_muls:12d}")

## Summary

| Step | API | What happens |
|------|-----|-------------|
| Define | `from_equations("dv/dt = ...")` | Parse ODE string, compile to Python bytecode + AST |
| Simulate | `neuron.step(I=...)` | Euler integration at `dt`, spike detection + reset |
| Compile | `compile_to_verilog(neuron)` | AST → Q8.8 fixed-point Verilog with LUT-based transcendentals |

Limitations:
- Q8.8 range $[-128, +128)$ — sufficient for most neuron dynamics but
  clips large values (e.g. HH conductances need scaling)
- Transcendental functions (`exp`, `tanh`, `sqrt`) use 16-entry
  piecewise-linear LUTs with ~1-2% accuracy
- Integer powers up to 8 supported; arbitrary exponents are not

The generated RTL is directly synthesisable with Yosys or Vivado.
See `sc-neurocore deploy model.nir --target ice40` for one-command
FPGA deployment.